In [1]:
import math
import random
from collections import defaultdict, Counter
from typing import List, Dict, Tuple, Optional

import numpy as np


class BayesianNGramMCMC:
    """
    Bayesian n-gram with interpolated backoff and MCMC over (phi_h, lambda_h, z_assignments).
    - Supports n=2 (bigram) and n=3 (trigram).
    - Vocabulary is inferred from data.
    - Context h is a tuple of length (n-1), using BOS (and optional BOS stacking) for padding.
    """

    def __init__(
        self,
        n: int = 2,
        alpha: float = 0.5,        # Dirichlet concentration for phi_h
        alpha0: float = 0.5,       # Dirichlet concentration for unigram root
        a_beta: float = 2.0,       # Beta(a,b) prior for lambda_h
        b_beta: float = 2.0,
        bos_id: int = -1,          # BOS token id (must not appear in real data)
        seed: int = 0
    ):
        assert n in (2, 3), "Only bigram (n=2) or trigram (n=3) supported."
        self.n = n
        self.alpha = alpha
        self.alpha0 = alpha0
        self.a_beta = a_beta
        self.b_beta = b_beta
        self.bos_id = bos_id
        self.rng = np.random.default_rng(seed)

        # Learned / sampled state
        self.vocab_: List[int] = []
        self.vsz_: int = 0

        # Context sets
        self.contexts_: set = set()      # all observed contexts of length n-1
        self.contexts_lower_: set = set()  # for backoff (length n-2) when n=3

        # Parameters (per MCMC state)
        # phi[h]   : np.array shape [V]  (Dirichlet sample)
        # lam[h]   : float in (0,1)      (Beta sample)
        # counts[h]: Counter[token] for z=0 assigned tokens at this h
        self.phi_: Dict[Tuple[int, ...], np.ndarray] = {}
        self.lam_: Dict[Tuple[int, ...], float] = {}
        self.counts_: Dict[Tuple[int, ...], Counter] = {}

        # For unigram root
        self.phi_root_: np.ndarray = None  # Dir(alpha0 * u)
        self.counts_root_: Counter = Counter()

        # Assignment z for each token occurrence (except positions with no full context)
        # We store per (seq_idx, position) -> z in {0,1}
        self.z_assign_: Dict[Tuple[int, int], int] = {}

        # Data index for efficient iteration
        self.data_index_: List[Tuple[int, int, Tuple[int, ...], int]] = []  # (seq_idx, pos, context, token_id)

    # ---------- Utilities ----------
    def _extract_vocab(self, sequences: List[List[int]]):
        vocab = set()
        for seq in sequences:
            for tok in seq:
                vocab.add(tok)
        if self.bos_id in vocab:
            raise ValueError("bos_id must be unique and not appear in actual tokens")
        vocab_list = sorted(vocab)
        self.vocab_ = vocab_list
        self.vsz_ = len(vocab_list)
        self._tok2idx = {t: i for i, t in enumerate(self.vocab_)}

    def _bos_pad(self, seq: List[int], k: int) -> List[int]:
        """Pad the beginning with k BOS tokens."""
        return [self.bos_id] * k + seq

    def _build_data_index(self, sequences: List[List[int]]):
        """
        Build list of (seq_idx, pos, context_tuple, token_id) for positions having a full (n-1)-length context.
        Also collect the set of contexts for parameterization.
        """
        self.data_index_.clear()
        self.contexts_.clear()
        self.contexts_lower_.clear()

        k = self.n - 1
        for i, seq in enumerate(sequences):
            s = self._bos_pad(seq, k)
            # positions in original indexing: t runs from k to k+len(seq)-1
            for t in range(k, len(s)):
                ctx = tuple(s[t-k:t])    # length k context
                w = s[t]
                self.data_index_.append((i, t, ctx, w))
                self.contexts_.add(ctx)
                if self.n == 3:
                    # lower-order (bigram) context for backoff is last (k-1) tokens of ctx
                    self.contexts_lower_.add(ctx[1:])

        # Ensure backoff includes the empty context (root)
        # root is represented as empty tuple ()
        if self.n == 2:
            pass  # bigram backoff is root
        elif self.n == 3:
            self.contexts_lower_.add(())  # root for bigram backoff

    def _init_params(self):
        """Randomly initialize phi, lambda, counts, and z assignments."""
        # Root phi
        self.phi_root_ = self.rng.dirichlet([self.alpha0 / self.vsz_] * self.vsz_)
        self.counts_root_.clear()

        # Higher-order phi & lambda
        self.phi_.clear()
        self.lam_.clear()
        self.counts_.clear()

        for h in self.contexts_:
            self.phi_[h] = self.rng.dirichlet([self.alpha / self.vsz_] * self.vsz_)
            self.lam_[h] = self.rng.beta(self.a_beta, self.b_beta)
            self.counts_[h] = Counter()

        if self.n == 3:
            # for bigram layer as well (backoff of trigram)
            for h2 in self.contexts_lower_:
                if h2 not in self.phi_:
                    self.phi_[h2] = self.rng.dirichlet([self.alpha / self.vsz_] * self.vsz_)
                    self.lam_[h2] = self.rng.beta(self.a_beta, self.b_beta)
                    self.counts_[h2] = Counter()

        # Random z assignments and initial counts
        self.z_assign_.clear()
        for (seq_idx, t, h, w) in self.data_index_:
            # sample z in {0,1} with 0.5 prob
            z = 0 if random.random() < 0.5 else 1
            self.z_assign_[(seq_idx, t)] = z
            if z == 0:
                # direct to phi_h
                if w != self.bos_id:   # bos shouldn't be counted as a vocab token; usually w != bos here
                    self.counts_[h][w] += 1
            else:
                # backoff path: recursively ends at root; we don't store counts along the recursion,
                # because those are accounted at re-sampling time; only root gets counts if we wanted
                pass

    # ---------- Probabilities ----------
    def _phi_prob(self, h: Tuple[int, ...], w: int) -> float:
        """Return phi_h(w) (if h==() use root)."""
        if len(h) == 0:
            return self.phi_root_[self._tok2idx[w]]
        return self.phi_[h][self._tok2idx[w]]

    def _p_backoff(self, h: Tuple[int, ...], w: int) -> float:
        """Recursive backoff probability p(w | bo(h)), ending at root unigram."""
        # If at root
        if len(h) == 0:
            return self.phi_root_[self._tok2idx[w]]
        # otherwise mixture at lower level:
        lam = self.lam_[h]
        # high-order component at current h
        ph = self.phi_[h][self._tok2idx[w]]
        # backoff component
        pbo = self._p_backoff(h[1:], w)  # drop leftmost, recurse
        return (1 - lam) * ph + lam * pbo

    def _p_full(self, h: Tuple[int, ...], w: int) -> float:
        """Full p(w | h) = (1 - lambda_h) phi_h(w) + lambda_h p(w | bo(h)).  Root uses phi_root only."""
        if len(h) == 0:
            return self.phi_root_[self._tok2idx[w]]
        lam = self.lam_[h]
        ph = self.phi_[h][self._tok2idx[w]]
        pbo = self._p_backoff(h[1:], w)  # backoff to lower order
        return (1 - lam) * ph + lam * pbo

    # ---------- Gibbs steps ----------
    def _sample_phi_all(self):
        """Sample all phi from Dirichlet posteriors given current z-assign counts."""
        # Root
        # （可选：若希望 root 也参与 z=0/1 的显式分配，可累积 root 的直出计数；此处简化：root只按先验采样）
        self.phi_root_ = self.rng.dirichlet([self.alpha0 / self.vsz_] * self.vsz_)

        # Others (both layer(s))
        for h, cnt in self.counts_.items():
            alphas = np.full(self.vsz_, self.alpha / self.vsz_, dtype=np.float64)
            # add counts to the corresponding token dims
            for tok, c in cnt.items():
                alphas[self._tok2idx[tok]] += c
            self.phi_[h] = self.rng.dirichlet(alphas)

    def _sample_lambda_all(self):
        """Sample all lambda_h ~ Beta(a + B_h, b + D_h) from current z-assignments."""
        # compute (B_h, D_h) by scanning z_assign_
        B = defaultdict(int)
        D = defaultdict(int)
        for (seq_idx, t, h, w) in self.data_index_:
            z = self.z_assign_[(seq_idx, t)]
            if len(h) == 0:
                continue  # root has no lambda
            if z == 1:
                B[h] += 1
            else:
                D[h] += 1
        for h in self.phi_.keys():
            if len(h) == 0:
                continue
            a_post = self.a_beta + B[h]
            b_post = self.b_beta + D[h]
            # avoid degenerate
            a_post = max(a_post, 1e-6)
            b_post = max(b_post, 1e-6)
            self.lam_[h] = self.rng.beta(a_post, b_post)

    def _resample_z_all(self):
        """Resample z_t for each token occurrence."""
        for (seq_idx, t, h, w) in self.data_index_:
            # remove current z's effect on counts
            prev_z = self.z_assign_[(seq_idx, t)]
            if prev_z == 0:
                self.counts_[h][w] -= 1
                if self.counts_[h][w] <= 0:
                    del self.counts_[h][w]

            # compute current probabilities
            # P(z=0) ∝ (1 - λ_h) * φ_h(w)
            # P(z=1) ∝ λ_h * p(w | bo(h))
            if len(h) == 0:
                # root has no z; treat as z=0 always (directly from root phi)
                self.z_assign_[(seq_idx, t)] = 0
                continue

            lam = self.lam_[h]
            phi_hw = self._phi_prob(h, w)
            p_back = self._p_backoff(h[1:], w)

            p0 = (1 - lam) * max(phi_hw, 1e-12)
            p1 = lam * max(p_back, 1e-12)
            s = p0 + p1
            if s == 0.0:
                # extremely rare; fallback to 0.5/0.5
                p0_ = 0.5
            else:
                p0_ = p0 / s

            z = 0 if random.random() < p0_ else 1
            self.z_assign_[(seq_idx, t)] = z

            # add effect back
            if z == 0:
                self.counts_[h][w] += 1

    # ---------- Public API ----------
    def fit(
        self,
        sequences: List[List[int]],
        iters: int = 2000,
        burnin: int = 500,
        thin: int = 5,
        verbose: bool = True
    ):
        """
        Run MCMC. After finishing, self.post_q_ stores averaged posterior predictive q(w|h).
        """
        # Prepare
        self._extract_vocab(sequences)
        self._build_data_index(sequences)
        self._init_params()

        # Storage for posterior predictive averaging
        accum = defaultdict(lambda: np.zeros(self.vsz_, dtype=np.float64))
        samples_kept = 0

        for it in range(1, iters + 1):
            # Gibbs: sample phi, lambda, then z
            self._sample_phi_all()
            self._sample_lambda_all()
            self._resample_z_all()

            # Collect posterior predictive after burn-in & thinning
            if it > burnin and ((it - burnin) % thin == 0):
                # accumulate q(w|h) for all contexts (including root)
                # root
                accum[()][...] += self.phi_root_
                # others
                for h in self.phi_.keys():
                    # compute full mixture p(.|h)
                    if len(h) == 0:
                        accum[h][...] += self.phi_root_
                    else:
                        lam = self.lam_[h]
                        # backoff probabilities vectorized via recursion per token
                        # (for simplicity loop over V; V不会太大时可接受；若V很大可缓存/并行)
                        probs = np.zeros(self.vsz_, dtype=np.float64)
                        for i_tok, tok in enumerate(self.vocab_):
                            ph = self.phi_[h][i_tok]
                            pbo = self._p_backoff(h[1:], tok)
                            probs[i_tok] = (1 - lam) * ph + lam * pbo
                        accum[h][...] += probs
                samples_kept += 1

            if verbose and (it % max(50, iters // 20) == 0):
                print(f"[MCMC] iter {it}/{iters}  kept={samples_kept}")

        # Normalize to get posterior means
        self.post_q_: Dict[Tuple[int, ...], Dict[int, float]] = {}
        if samples_kept == 0:
            raise RuntimeError("No posterior samples kept. Increase iters or reduce burnin/thin.")

        for h, vec in accum.items():
            avg = vec / samples_kept
            # normalize safety
            s = float(avg.sum())
            if s <= 0:
                avg = np.full_like(avg, 1.0 / self.vsz_)
            else:
                avg = avg / s
            self.post_q_[h] = {tok: float(avg[self._tok2idx[tok]]) for tok in self.vocab_}

        return self

    def posterior_q(self) -> Dict[Tuple[int, ...], Dict[int, float]]:
        """
        Return posterior predictive q_MCMC(w|h) as nested dict: q[h][token] = prob.
        Includes root context () and all seen contexts.
        """
        return self.post_q_

    def p_cond(self, context: Tuple[int, ...], token: int) -> float:
        """
        Convenience: query posterior q_MCMC(token | context). If unseen context,
        fallback to closest backoff / root uniform.
        """
        if hasattr(self, "post_q_") and context in self.post_q_:
            return self.post_q_[context].get(token, 0.0)
        # backoff fallback
        h = context
        while len(h) > 0:
            h = h[1:]
            if h in getattr(self, "post_q_", {}):
                return self.post_q_[h].get(token, 0.0)
        # root fallback
        if hasattr(self, "post_q_") and () in self.post_q_:
            return self.post_q_[()].get(token, 0.0)
        # uniform last resort
        return 1.0 / max(1, self.vsz_)

## expr 24

In [2]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.2-1B")

/home/xw3763/miniforge3/envs/torch/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
#!/usr/bin/env python3
from itertools import product
from fractions import Fraction

all_tokens_list = []

OPS = ['+', '-', '*', '/']

def eval_no_paren(nums, ops):
    """按 * / 高于 + - 的标准优先级，用 Fraction 精确计算"""
    a, b, c, d = [Fraction(x, 1) for x in nums]
    o1, o2, o3 = ops

    # 先处理 * /，把表达式分解成若干项（带符号），每项内部只有 * 或 /
    terms = []
    cur = a
    sign = 1  # 记录当前项之前的 +/-
    for op, n in [(o1, b), (o2, c), (o3, d)]:
        if op == '*':
            cur *= n
        elif op == '/':
            if n == 0:
                return None  # 除零非法
            cur /= n
        elif op == '+':
            terms.append(sign * cur)
            cur = n
            sign = 1
        elif op == '-':
            terms.append(sign * cur)
            cur = n
            sign = -1
    terms.append(sign * cur)

    return sum(terms, Fraction(0, 1))


target = Fraction(24, 1)
count = 0
for digits in product(range(10), repeat=4):          # 10^4 组合
    for ops in product(OPS, repeat=3):               # 4^3 组合
        val = eval_no_paren(digits, ops)
        if val is None:
            continue
        if val == target:
            expr = f"{digits[0]}{ops[0]}{digits[1]}{ops[1]}{digits[2]}{ops[2]}{digits[3]}"
            tokenized_expr = tokenizer.encode(expr, add_special_tokens=False)
            all_tokens_list.append(tokenized_expr)
            count += 1
                
print(f"Total: {count}")

Total: 3253


In [5]:
import torch
all_tokens_list = torch.tensor(all_tokens_list)
torch.save(all_tokens_list, "/data1/xw3763/project/gflow/ChemGFN/data/24_points/buffer_24.pt")

In [5]:
# 1) 训练一个 Bigram (n=2) 的 Bayesian n-gram MCMC
model = BayesianNGramMCMC(
    n=2,             # 改成3即可 Trigram
    alpha=0.5,
    alpha0=0.5,
    a_beta=2.0,
    b_beta=2.0,
    bos_id=-1,
    seed=42
)
model.fit(
    all_tokens_list,
    iters=1500,
    burnin=300,
    thin=5,
    verbose=True
)

[MCMC] iter 75/1500  kept=0
[MCMC] iter 150/1500  kept=0
[MCMC] iter 225/1500  kept=0
[MCMC] iter 300/1500  kept=0
[MCMC] iter 375/1500  kept=15
[MCMC] iter 450/1500  kept=30
[MCMC] iter 525/1500  kept=45
[MCMC] iter 600/1500  kept=60
[MCMC] iter 675/1500  kept=75
[MCMC] iter 750/1500  kept=90
[MCMC] iter 825/1500  kept=105
[MCMC] iter 900/1500  kept=120
[MCMC] iter 975/1500  kept=135
[MCMC] iter 1050/1500  kept=150
[MCMC] iter 1125/1500  kept=165
[MCMC] iter 1200/1500  kept=180
[MCMC] iter 1275/1500  kept=195
[MCMC] iter 1350/1500  kept=210
[MCMC] iter 1425/1500  kept=225
[MCMC] iter 1500/1500  kept=240


In [6]:
q = model.posterior_q()  # dict[(ctx_tuple) -> dict[token_id -> prob]]

root = ()
print("\nPosterior q(token | BOS):")
for tok, prob in sorted(q[root].items(), key=lambda x: -x[1])[:10]:
    print(f"  token={tok:<4d}  p={prob:.4f}")


Posterior q(token | BOS):
  token=20    p=0.0865
  token=23    p=0.0865
  token=9     p=0.0844
  token=16    p=0.0837
  token=15    p=0.0738
  token=17    p=0.0724
  token=24    p=0.0717
  token=12    p=0.0712
  token=19    p=0.0677
  token=14    p=0.0662
